<a href="https://colab.research.google.com/github/oliviaberset/2026_Intro_Python/blob/main/live/part-I/1.7-oop-for-environmental-systems-environmental-exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> Notebook-friendly copy of `part-I/1.7-oop-for-environmental-systems-environmental-exercises.ipynb`, generated by `tools/make_live.py`. Edit the book notebook, not this file.

In [161]:
# --- environment setup (generated, not part of the lesson) ---
# Colab and Kaggle do not ship every package this notebook imports.
# This is a no-op in an environment that is already set up.
import importlib.util
import subprocess
import sys

for module, package in {"pooch": "pooch"}.items():
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

# Environmental Exercises

**ℹ️ How to use these exercises**

Attempt each from scratch in the empty code cell. Type-hint your methods, give each class a useful `__repr__` where it helps, validate input with exceptions, reserve assert for invariants, and keep units in attribute/variable names.

## Exercise 1: Validated earthquake events — a real dataset

The [USGS](https://earthquake.usgs.gov/) publishes a feed of every earthquake it records; this exercise uses an archived extract of one month of it, January 2023.

Combine both halves of this subchapter: a class that carries state, and exceptions that refuse to let it be built from bad data.

In [162]:
# Pre-supplied: download and cache one month of the USGS earthquake feed (January 2023).
import pooch

quakes_path = pooch.retrieve(
    url="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/data/part-I/usgs_earthquakes_2023_01.csv",
    known_hash="sha256:14e6100a41c55772f73a34daf1cd6ea38ba0fa49a86bb62a3930868614975df2",
    fname="usgs_earthquakes_2023_01.csv",
    path=pooch.os_cache("mlees"),
)

**Step 1.** Read `quakes_path` with pandas, keeping only the `place`, `mag`, and `depth` columns. Print the resulting shape.

In [163]:
# Your solution here
# Hint: pd.read_csv(quakes_path)[["place", "mag", "depth"]]
import pandas as pd
pd.read_csv(quakes_path)[["place","mag","depth"]]

,place,mag,depth
0,"103 km S of ?urrieq, Malta",5.3,10.300
1,"95 km SE of Ishinomaki, Japan",4.5,26.488
2,"105 km ENE of Port Blair, India",5.0,107.401
3,"258 km ESE of Tadine, New Caledonia",4.5,10.000
4,"152 km S of Hotan, China",5.0,10.000
...,...,...,...
624,South Sandwich Islands region,5.2,44.616
625,"36 km S of Jurm, Afghanistan",4.8,187.282
626,"157 km WNW of Pathein, Myanmar",4.7,39.850
627,"51 km SSW of Panguna, Papua New Guinea",5.4,35.000


In [164]:
print(f"Shape : {pd.read_csv(quakes_path)[["place","mag","depth"]].shape}")

Shape : (629, 3)


**Step 2.** Define an `EarthquakeEvent` class with instance attributes `place: str`, `magnitude: float`, and `depth_km: float`; a `__repr__` showing all three; and a method `is_shallow(self, threshold_km=70.0) -> bool` returning whether `depth_km` is below the threshold. In `__init__`, raise `ValueError` if `magnitude` or `depth_km` is negative — this feed lists only events of magnitude 4.5 or larger, with depth measured below the surface, so a negative value is a data-entry error; it comes from outside the code, not from your own arithmetic, so it is an exception, not an assert.

In [165]:
# Your solution here
# Hint: def __init__(self, place: str, magnitude: float, depth_km: float) -> None:
#           if magnitude < 0.0 or depth_km < 0.0: raise ValueError(...)
# Hint: def is_shallow(self, threshold_km: float = 70.0) -> bool:
#           return self.depth_km < threshold_km

class EarthquakeEvent:
  def __init__(self, place: str, magnitude: float, depth_km: float) -> None :
    if magnitude < 0.0 or depth_km < 0.0:
      raise ValueError(f"data-entry error: magnitude={magnitude}, depth={depth_km}")
    self.place = place
    self.magnitude = magnitude
    self.depth_km = depth_km
  def __repr__(self) -> str:
    return f"{self.place}, {self.magnitude}, {self.depth_km}"
  def is_shallow(self, threshold_km: float=70.0):
    return self.depth_km < threshold_km

**Step 3.** Confirm the validation works: build one valid `EarthquakeEvent` and print it, then show that `EarthquakeEvent("test", -1.0, 10.0)` raises `ValueError`.

In [166]:
# Your solution here
# Hint: wrap the invalid construction in try/except ValueError

malta = EarthquakeEvent("Malta", 5.3, 10.300)
test = EarthquakeEvent("test", -1.0, 10.0)    # fonctionne bien, on a le message : ValueError: data-entry error

ValueError: data-entry error: magnitude=-1.0, depth=10.0

**Step 4.** The January feed is clean, so first append two deliberately corrupted rows, the kind of typo the constructor exists to catch:

```python
bad_rows = pd.DataFrame({
    "place": ["typo: sign dropped", "typo: sign dropped"],
    "mag": [-4.5, 4.5],
    "depth": [10.0, -35.0],
})
quakes = pd.concat([quakes, bad_rows], ignore_index=True)
```

Then loop over the three columns with `zip`, building one `EarthquakeEvent` per row inside a `try/except ValueError`. Collect the successfully built events in a list, and count how many rows were rejected instead of stopping the loop.

In [ ]:
bad_rows = pd.DataFrame({
    "place": ["typo: sign dropped", "typo: sign dropped"],
    "mag": [-4.5, 4.5],
    "depth": [10.0, -35.0],
})
quakes = pd.concat([quakes, bad_rows], ignore_index=True)

In [ ]:
quakes = pd.read_csv(quakes_path)
quakes.head()

In [ ]:
quakes.shape

In [ ]:
bad_rows.head()

In [ ]:
# Your solution here
# Hint: bad_rows = pd.DataFrame({"place": [...], "mag": [...], "depth": [...]})
#       quakes = pd.concat([quakes, bad_rows], ignore_index=True)
# Hint: events = []
#       n_rejected = 0
#       for place, mag, depth in zip(quakes["place"], quakes["mag"], quakes["depth"]):
#           try:
#               events.append(EarthquakeEvent(place, mag, depth))
#           except ValueError:
#               n_rejected += 1

import pandas as pd

bad_rows = pd.DataFrame({"place":["typo:sign dropped", "typo:sign dropped"], "mag": [-4.5, 4.5], "depth":[10.5, -35.0]})
quakes = pd.concat([quakes, bad_rows], ignore_index=True)

events = []
n_rejected = 0
for place, mag, depth in zip(quakes["place"], quakes["mag"], quakes["depth"]):
  try:
    events.append(EarthquakeEvent(place, mag, depth))
  except ValueError:
    n_rejected += 1

print(len(events), n_rejected)


**Step 5.** Using `is_shallow()`, count how many events in your list are shallow. Separately, find the single deepest event with an explicit loop that tracks a running "deepest so far" variable — the same pattern `StationNetwork.coldest_station` uses in the lecture, not `max()`.

In [ ]:
# Your solution here
# Hint: n_shallow = 0
#       for event in events:
#           if event.is_shallow():
#               n_shallow += 1
# Hint: deepest = None
#       for event in events:
#           if deepest is None or event.depth_km > deepest.depth_km:
#               deepest = event

n_shallow = 0
for event in events:
  if event.is_shallow():
    n_shallow +=1
print(n_shallow)

deepest_so_far = None
for event in events:
  if deepest_so_far is None or event.depth_km > deepest_so_far.depth_km :
    deepest_so_far = event
print(f"Single deepest event: {deepest_so_far}")

**Step 6.** Print a short summary: the number of events built, the number of rows rejected, the number of shallow events, and the deepest event's repr.

In [ ]:
# Your solution here
# Hint: print(len(events), "events,", n_rejected, "rejected,", n_shallow, "shallow")
#       print(deepest)

print(f"Summary:\n - Number of events built: {len(events)}\n - Number of rows rejected: {n_rejected}\n - Number of shallow events: {n_shallow}\n - Deepest event: {deepest_so_far}")

## Exercise 2: The solar system, continued

Exercise 2 of [1.2](1.2-data-structures-and-control-flow-environmental-exercises.ipynb) left the solar system
as a dictionary: `planet_masses_1e24kg`, mapping a name to a mass in units of 10^24 kg. A
dictionary keeps the name and the mass together only by convention — pass the mass somewhere on
its own and the name is gone, which is exactly what went wrong in that exercise's bonus question.

Now that classes are available, finish the job: give a planet a type, so that its name and its mass
travel as one object and the comparison you want to make becomes a method on it.

The dictionary is rebuilt below so this exercise stands on its own.

In [167]:
# Pre-supplied: the dictionary from Exercise 2 of 1.2's environmental exercises, rebuilt here.
planet_masses_1e24kg = {
    "mercury": 0.330,
    "venus": 4.87,
    "earth": 5.97,
    "mars": 0.642,
    "jupiter": 1898.0,
    "saturn": 568.0,
    "uranus": 86.8,
    "neptune": 102.0,
}

**Q1) Write a class `Planet` with two attributes, `name` and `mass_1e24kg`.**

Create one instance for Earth (5.97) and one for Jupiter (1898), then print each one's name and
mass to check they were stored.

Hint: give `__init__` type annotations, as 1.7's lecture does — `name: str`, `mass_1e24kg: float`.
They are not enforced at runtime, but they are the only place the unit is written down.

In [186]:
# Q1: define the Planet class here, then create earth and jupiter

class Planet:
  def __init__(self, name: str, mass_1e24kg: float)-> None:
    if mass_1e24kg < 0.0:
      raise ValueError(f"Implausible mass: mass={mass_1e24kg} 1e24kg")
    self.name = name
    self.mass_1e24kg = mass_1e24kg

earth = Planet("earth", 5.97)
print(earth.name, earth.mass_1e24kg)

jupiter = Planet("jupiter", 1898.0)
print(jupiter.name, jupiter.mass_1e24kg)

earth 5.97
jupiter 1898.0


**Q2) Add a method `is_light` that compares a planet against Jupiter.**

It should report `True` when the planet is strictly lighter than Jupiter, `False` when it is
strictly heavier, and the string `"same mass!"` when the two are equal. Ask it whether Earth is
lighter than Jupiter, and whether Jupiter is lighter than itself.

In [222]:
class Planet:
  def __init__(self, name: str, mass_1e24kg: float)-> None:
    if mass_1e24kg < 0.0:
      raise ValueError(f"Implausible mass: mass={mass_1e24kg} 1e24kg")
    self.name = name
    self.mass_1e24kg = mass_1e24kg
  def is_light(self) -> bool:
    if self.mass_1e24kg < 1898.0:
      return True
    elif self.mass_1e24kg == 1898.0:
      return "same mass!"
    else:
      return False
earth = Planet(earth.name, 5.97)
jupiter = Planet(jupiter.name, 1898.0)

print(f"{earth.name}: {earth.is_light()}")
print(f"{jupiter.name}: {jupiter.is_light()}")

earth: True
jupiter: same mass!


**Q3) Now fix what Q2 asked you to build.**

The method you just wrote has two problems that 1.7's own material names directly.

1. It *prints* its answer instead of returning it, so a caller cannot use the result —
   `if earth.is_light():` will not do what you expect.
2. It returns a `bool` on two branches and a `str` on the third, so the caller has to test the
   type of the answer before trusting it.

Rewrite it as `compare_to_jupiter()` returning one of three strings — `"lighter"`, `"heavier"`,
`"same"` — and have the caller do the printing. Then loop over every planet in
`planet_masses_1e24kg`, build a `Planet` for each, and print the verdict for all eight.

In [ ]:
# Q3: rewrite it as compare_to_jupiter, then report all eight planets
class Planet:
  def __init__(self, name: str, mass_1e24kg: float)-> None:
    if mass_1e24kg < 0.0:
      raise ValueError(f"Implausible mass: mass={mass_1e24kg} 1e24kg")
    self.name = name
    self.mass_1e24kg = mass_1e24kg
  def compare_to_jupiter(self)-> bool:




**Q4) Give the class the invariant it is missing.**

A negative mass is not a planet. Raise a `ValueError` from `__init__` when `mass_1e24kg` is not
strictly positive, and show both that a valid planet still constructs and that
`Planet("nonsense", -1.0)` fails. Add a `__repr__` so a `Planet` prints readably.

This is the same shape as Exercise 1's `EarthquakeEvent`: validate at construction, so that every
object that exists is one you can trust.

In [ ]:
# Q4: validate in __init__ and add __repr__, then show both outcomes

**ℹ️ What you just did**

You turned a dictionary into a type. The name and the mass are now one object rather than a key
and a value that happen to line up, the comparison is a method rather than a loose function that
has to be handed the right dictionary, and the constructor refuses to build a planet that could not
exist.

The Q2-to-Q3 step is worth a second look. The method Q2 asks for prints instead of returning and
changes its return type between branches. Both are easy to write and hard to use, and neither shows up until
something calls the method — which is why 1.7 spends its second half on making failures visible
early rather than on making code shorter.